# NYC Flight Delay Analysis ✈️

**A Google Colab + Python data-analysis portfolio project**

**Questions:** How often were flights late? How did late-arrival rates differ across months, NYC airports, airlines, departure hours, and routes?

**Data:** [OpenIntro NYC Flights](https://www.openintro.org/data/index.php?data=nycflights) — a random sample of **32,735 flights departing JFK, LGA, or EWR in 2013**.

**Definition:** A flight is **late** when its recorded arrival delay is **more than 15 minutes**. Records without an arrival-delay value are excluded from rate denominators and counted separately. The dataset is historical and these results are descriptive, not current operational predictions.

## 1. Set up the analysis

Colab already includes the usual Python data-science libraries. Run cells in order with **Runtime → Run all**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 25)
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
print("Environment ready.")

## 2. Load and validate the dataset

The dataset originally comes from **OpenIntro**. For reliable loading, this project also includes the same 32,735-row CSV under `data/nycflights.csv`. Colab loads the repository-hosted copy over HTTPS; no manual CSV upload is needed.

In [ ]:
# Public dataset snapshot included with this GitHub project for reliable Colab access.
SOURCE_URL = "https://raw.githubusercontent.com/WINNER03/GOOGLE-DATA-ANALYSTIC/main/projects/nyc-flight-delay-analysis/data/nycflights.csv"
raw = pd.read_csv(SOURCE_URL)
# The Rdatasets export includes a row-index column that is not a flight feature.
raw = raw.drop(columns=["rownames"], errors="ignore")

required = {
    "year", "month", "day", "dep_delay", "arr_delay",
    "carrier", "origin", "dest", "hour"
}
missing_columns = required - set(raw.columns)
if missing_columns:
    raise ValueError(f"Unexpected dataset schema. Missing columns: {sorted(missing_columns)}")

print(f"Rows: {len(raw):,} | Columns: {len(raw.columns)}")
display(raw.head())
display(raw.dtypes.rename("dtype").to_frame())

## 3. Data quality and cleaning

An absent arrival delay does **not** prove that the flight was canceled. We keep missing records visible instead of treating them as on-time flights.

In [ ]:
missing = (
    raw.isna().sum()
    .rename("missing_rows")
    .to_frame()
    .assign(missing_pct=lambda t: (100 * t["missing_rows"] / len(raw)).round(2))
    .sort_values("missing_rows", ascending=False)
)
display(missing.head(12))

flights = raw.copy()
flights["date"] = pd.to_datetime(flights[["year", "month", "day"]], errors="coerce")
flights["route"] = flights["origin"].astype(str) + " → " + flights["dest"].astype(str)
flights["hour"] = pd.to_numeric(flights["hour"], errors="coerce")

arrivals = flights.loc[flights["arr_delay"].notna()].copy()
arrivals["late_15"] = arrivals["arr_delay"] > 15

print(f"Total flight records: {len(flights):,}")
print(f"Records with known arrival delay: {len(arrivals):,}")
print(f"Records without known arrival delay: {len(flights) - len(arrivals):,}")
assert len(arrivals) + flights["arr_delay"].isna().sum() == len(flights)

## 4. Calculate headline KPIs

The denominator for the late-arrival percentage includes **only flights with a recorded arrival delay**.

In [ ]:
observed_n = len(arrivals)
late_n = int(arrivals["late_15"].sum())
late_rate = 100 * late_n / observed_n

kpis = pd.DataFrame(
    {
        "Metric": [
            "All flight records",
            "Flights with observed arrival delay",
            "Records missing arrival delay",
            "Late arrivals (>15 min)",
            "Late-arrival rate (%)",
            "Mean arrival delay (min)",
            "Median arrival delay (min)",
        ],
        "Value": [
            len(flights),
            observed_n,
            len(flights) - observed_n,
            late_n,
            round(late_rate, 2),
            round(arrivals["arr_delay"].mean(), 2),
            round(arrivals["arr_delay"].median(), 2),
        ],
    }
)
display(kpis)
kpis.to_csv(OUT / "headline_kpis.csv", index=False)

## 5. Reusable aggregation function

Compute comparable late rates and delay summaries for each group. Check sample sizes before drawing conclusions.

In [ ]:
def summarize_by(data, column):
    return (
        data.groupby(column, dropna=False)
        .agg(
            flights=("late_15", "size"),
            late_arrivals=("late_15", "sum"),
            late_rate_pct=("late_15", lambda s: 100 * s.mean()),
            mean_arr_delay_min=("arr_delay", "mean"),
            median_arr_delay_min=("arr_delay", "median"),
        )
        .reset_index()
    )

monthly = summarize_by(arrivals, "month").sort_values("month")
airports = summarize_by(arrivals, "origin").sort_values("late_rate_pct")
carriers = summarize_by(arrivals, "carrier")
hours = summarize_by(arrivals.dropna(subset=["hour"]), "hour").sort_values("hour")
routes = summarize_by(arrivals, "route")

display(monthly)
display(airports)
display(carriers.sort_values("flights", ascending=False).head(10))

## 6. Chart: Delays by month and airport

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(data=monthly, x="month", y="late_rate_pct", marker="o", ax=axes[0])
axes[0].set(title="Late-arrival rate by month (2013)", xlabel="Month", ylabel="Late arrivals (%)")
axes[0].set_xticks(range(1, 13))

sns.barplot(data=airports, x="origin", y="late_rate_pct", ax=axes[1])
axes[1].set(title="Late-arrival rate by NYC origin", xlabel="Departure airport", ylabel="Late arrivals (%)")

fig.tight_layout()
fig.savefig(OUT / "monthly_and_airport_delay.png", dpi=170, bbox_inches="tight")
plt.show()

## 7. Chart: Airlines and departure hours

For the airline comparison, require at least **100 observed flights** in this sample. This is a descriptive filter, not a statistical significance test.

In [ ]:
carrier_min_flights = 100
carrier_view = (
    carriers.query("flights >= @carrier_min_flights")
    .sort_values("late_rate_pct", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=carrier_view, y="carrier", x="late_rate_pct", ax=axes[0])
axes[0].set(title="Airlines: late arrivals (>15 min)", xlabel="Late arrivals (%)", ylabel="Carrier code")

sns.lineplot(data=hours, x="hour", y="late_rate_pct", marker="o", ax=axes[1])
axes[1].set(title="Late arrivals by departure-hour field", xlabel="Hour (0–23)", ylabel="Late arrivals (%)")
axes[1].set_xticks(range(0, 24, 3))

fig.tight_layout()
fig.savefig(OUT / "airline_and_hour_delay.png", dpi=170, bbox_inches="tight")
plt.show()

display(carrier_view[["carrier", "flights", "late_rate_pct"]])

## 8. Busiest routes and departure/arrival relationship

Route comparisons are limited to routes with at least **30 observed arrivals**. The correlation below is an association, not proof of causation.

In [ ]:
route_min_flights = 30
busy_routes = (
    routes.query("flights >= @route_min_flights")
    .sort_values("flights", ascending=False)
    .head(12)
)
display(busy_routes)

paired = arrivals[["dep_delay", "arr_delay"]].dropna()
correlation = paired["dep_delay"].corr(paired["arr_delay"])
plot_sample = paired.sample(n=min(6000, len(paired)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=busy_routes, y="route", x="flights", ax=axes[0])
axes[0].set(title="12 busiest routes in sample", xlabel="Observed arrivals", ylabel="Route")

axes[1].scatter(plot_sample["dep_delay"], plot_sample["arr_delay"], s=8, alpha=0.16)
axes[1].set(
    title=f"Departure vs arrival delay (Pearson r={correlation:.2f})",
    xlabel="Departure delay (minutes)",
    ylabel="Arrival delay (minutes)",
)
fig.tight_layout()
fig.savefig(OUT / "routes_and_delay_relationship.png", dpi=170, bbox_inches="tight")
plt.show()

## 9. Export cleaned summary tables

Charts are saved as PNG files and grouped metrics as CSV files under **outputs/** in the Colab session. Download those files from Colab's file sidebar if you want to retain them.

In [ ]:
for filename, table in {
    "monthly_summary.csv": monthly,
    "airport_summary.csv": airports,
    "carrier_summary.csv": carriers,
    "hourly_summary.csv": hours,
    "route_summary.csv": routes,
}.items():
    table.to_csv(OUT / filename, index=False)

print("Files ready:")
for path in sorted(OUT.iterdir()):
    print(f" - {path}")

## 10. Data-backed takeaways and caveats

**Check actual results above before publishing conclusions.** Use the output below to generate initial talking points.

In [ ]:
best_month = monthly.loc[monthly["late_rate_pct"].idxmin()]
worst_month = monthly.loc[monthly["late_rate_pct"].idxmax()]
best_origin = airports.loc[airports["late_rate_pct"].idxmin()]
worst_origin = airports.loc[airports["late_rate_pct"].idxmax()]

print(f"Of {observed_n:,} flights with known arrival delay, {late_rate:.1f}% arrived >15 min late.")
print(f"Lowest monthly late rate: month {int(best_month['month'])} ({best_month['late_rate_pct']:.1f}%).")
print(f"Highest monthly late rate: month {int(worst_month['month'])} ({worst_month['late_rate_pct']:.1f}%).")
print(f"Lowest origin late rate: {best_origin['origin']} ({best_origin['late_rate_pct']:.1f}%).")
print(f"Highest origin late rate: {worst_origin['origin']} ({worst_origin['late_rate_pct']:.1f}%).")
print(f"Departure/arrival delay correlation on available pairs: {correlation:.2f}.")

### Interpretation safeguards

- This is **a sample of NYC departures in 2013**, not all US flights or current schedules.
- We **exclude missing arrival-delay values** from late-rate denominators; do not label all such records as cancellations.
- Early arrivals have negative delay values. Late means **strictly greater than 15 minutes**, not 15 or more.
- Carrier/airport comparisons are **unadjusted** for weather, route mix, season, distance, or time of day.
- Differences and correlations are **descriptive**; do not assume a cause.

**Next steps:** Join weather data, add confidence intervals, build an interactive dashboard, or investigate delay forecasting with an appropriate train/test split.